<a href="https://colab.research.google.com/github/NourEldin-Osama/AI-Notebooks/blob/main/CodingAgents/ClaudeCode.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Claude Code on Colab: Quick Start
This notebook installs and configures Claude Code CLI, Wetty, and ngrok so you can run Claude Code from a browser terminal in Google Colab.

## Requirements
1. A Colab runtime with internet access.
2. A `NGROK_AUTHTOKEN` secret in Colab.
3. A `OPENROUTER_API_KEY` secret in Colab.

## Get Your Tokens
1. ngrok account signup: [https://ngrok.com/signup](https://ngrok.com/signup)
2. ngrok authtoken page: [https://dashboard.ngrok.com/get-started/your-authtoken](https://dashboard.ngrok.com/get-started/your-authtoken)
3. OpenRouter signup: [https://openrouter.ai/](https://openrouter.ai/)
4. OpenRouter API keys page: [https://openrouter.ai/keys](https://openrouter.ai/keys)

## Notebook Flow
1. **Setup Claude Code, Wetty, Ngrok**: installs Claude Code CLI, Wetty, and pyngrok, then configures ngrok auth.
2. **Configure Claude Code**: writes `~/.claude.json` and `~/.claude/settings.json`, and routes model calls through OpenRouter-compatible Anthropic settings.
3. **Start Ngrok and Wetty**: opens a tunnel to port 3000 and starts Wetty so you can access a shell from the printed public URL.

## How to Use
1. Add required secrets in Colab (`NGROK_AUTHTOKEN`, `OPENROUTER_API_KEY`).
2. Run all cells from top to bottom once per runtime session.
3. Open the Wetty URL printed in the final cell.
4. In the terminal, run `claude`.

## Troubleshooting
1. If Wetty fails to start, rerun the last cell to recreate the tunnel and process.
2. If model/auth errors occur, verify your secrets and rerun the configuration cell.
3. If the runtime is restarted, rerun the notebook from the beginning.

## Security Notes
1. Treat the public ngrok URL as temporary and sensitive.
2. Rotate API keys/tokens if they were accidentally exposed.

In [ ]:
# @title Setup Claude Code, Wetty, Ngrok
import os
from google.colab import userdata

!curl -fsSL https://claude.ai/install.sh | bash

os.environ["PATH"] = f"{os.environ['HOME']}/.local/bin:{os.environ['PATH']}"

print("Installing Wetty...")
!npm install -g wetty@2.5.0 --quiet --no-progress > /dev/null 2>&1

print("Wetty successfully installed!")

print("Setting up ngrok...")
!uv pip install -q pyngrok

NGROK_AUTHTOKEN = userdata.get("NGROK_AUTHTOKEN")
# Now you need a ngrok authtoken. Get one free at https://ngrok.com, then run:
!ngrok config add-authtoken $NGROK_AUTHTOKEN
print("ngrok successfully configured!")

In [ ]:
# @title Configure Claude Code
import os
import json

# 1. Get API Key and define the model
OPENROUTER_API_KEY = userdata.get('OPENROUTER_API_KEY')
model = "openrouter/free" # @param ["stepfun/step-3.5-flash:free", "minimax/minimax-m2.5:free", "openrouter/free", "nvidia/nemotron-3-super-120b-a12b:free"]

# 2. Define directories
home_dir = os.path.expanduser("~")
claude_dir = os.path.join(home_dir, ".claude")
os.makedirs(claude_dir, exist_ok=True)

# 3. Skip Onboarding (.claude.json)
claude_json_path = os.path.join(home_dir, ".claude.json")
claude_json_data = {}

if os.path.exists(claude_json_path):
    with open(claude_json_path, "r") as f:
        try:
            claude_json_data = json.load(f)
        except json.JSONDecodeError:
            pass

claude_json_data["hasCompletedOnboarding"] = True

with open(claude_json_path, "w") as f:
    json.dump(claude_json_data, f, indent=2)

# 4. Configure Settings, API Key, and Model (.claude/settings.json)
settings_path = os.path.join(claude_dir, "settings.json")
settings_data = {}

if os.path.exists(settings_path):
    with open(settings_path, "r") as f:
        try:
            settings_data = json.load(f)
        except json.JSONDecodeError:
            pass

# Ensure 'env' key exists
if "env" not in settings_data:
    settings_data["env"] = {}

# Ensure 'enabledPlugins' key exists
if "enabledPlugins" not in settings_data:
    settings_data["enabledPlugins"] = {}

# Ensure 'extraKnownMarketplaces' key exists
if "extraKnownMarketplaces" not in settings_data:
    settings_data["extraKnownMarketplaces"] = {}

# Inject OpenRouter configs into the environment variables
settings_data["env"]["ANTHROPIC_API_KEY"] = ""
settings_data["env"]["ANTHROPIC_AUTH_TOKEN"] = OPENROUTER_API_KEY
settings_data["env"]["ANTHROPIC_BASE_URL"] = "https://openrouter.ai/api" # OpenRouter Anthropic-compatible URL
settings_data["env"]["API_TIMEOUT_MS"] = "3000000"

#
settings_data["env"]["CLAUDE_CODE_DISABLE_NONESSENTIAL_TRAFFIC"] = "1"
settings_data["env"]["CLAUDE_CODE_EXPERIMENTAL_AGENT_TEAMS"] = "1"

# Save the selected model
settings_data["env"]["ANTHROPIC_MODEL"] = model
settings_data["env"]["ANTHROPIC_DEFAULT_OPUS_MODEL"] = model
settings_data["env"]["ANTHROPIC_DEFAULT_SONNET_MODEL"] = model
settings_data["env"]["ANTHROPIC_DEFAULT_HAIKU_MODEL"] = model
settings_data["env"]["ANTHROPIC_SMALL_FAST_MODEL"] = model

# Plugins
settings_data["enabledPlugins"]["ralph-wiggum@claude-code-plugins"] = True

# Marketplaces
if "claude-code-plugins" not in settings_data["extraKnownMarketplaces"]:
    settings_data["extraKnownMarketplaces"]["claude-code-plugins"] = {}
settings_data["extraKnownMarketplaces"]["claude-code-plugins"]["source"] = {
    "source": "github",
    "repo": "anthropics/claude-code"
}

# Write back to settings.json
with open(settings_path, "w") as f:
    json.dump(settings_data, f, indent=2)

print("✅ Claude Code JSON settings configured successfully!")
print(f"🔹 Model routed to: {model}")
print(f"🔹 Base URL routed to: {settings_data['env']['ANTHROPIC_BASE_URL']}")

✅ Claude Code JSON settings configured successfully!
🔹 Model routed to: openrouter/free
🔹 Base URL routed to: https://openrouter.ai/api


In [ ]:
# @title Start Ngrok and Wetty
from pyngrok import ngrok

!pkill wetty

# Start ngrok tunnel on port 3000
# (We use 3000 because wetty use it)
public_url = ngrok.connect(3000).public_url
print(f"🚀 YOUR Wetty LINK: {public_url}")

# 4. Start Wetty
print("\nYou can open the terminal and type 'claude'")
print(">>> claude\n")
# 4. Start Wetty with a direct login shell
!wetty --port 3000 --base / --command /bin/bash &